# 05 - ML Modeling: Ensemble + Ablation Study
**ZHAW AI-Applications: Skin Lesion Classifier**

**Block 3: ML Numeric**

Trains and evaluates the ensemble ML pipeline:
- **Models**: Logistic Regression vs Random Forest vs XGBoost
- **Features**: CV probabilities + Metadata + NLP features
- **Ablation study**: Systematically ablate feature groups
- **Metrics**: ROC-AUC (primary) + F1-Macro (secondary)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CLASS_NAMES, HAM10000_CLASSES, NUM_CLASSES
print('Setup complete')

Loaded: train=7009 rows, val=1503 rows
Feature groups: cv_only(7d), metadata_only(4d), cv_meta(11d), all_features(21d)


## 1. Load Features

In [ ]:
from src.ml.features import load_processed_data, build_feature_matrix, get_cv_probs_from_checkpoint
from pathlib import Path

train_df = load_processed_data('train', Path('../data/processed'))
test_df  = load_processed_data('test',  Path('../data/processed'))
y_train  = train_df['label'].values
y_test   = test_df['label'].values
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# Get CV probabilities (requires trained model)
import os
cv_model_path = os.getenv('CV_MODEL_PATH', '../models/resnet50_best.pth')

if Path(cv_model_path).exists():
    print(f'Loading CV model from {cv_model_path}')
    cv_probs_train = get_cv_probs_from_checkpoint(train_df, cv_model_path, device='cpu')
    cv_probs_test  = get_cv_probs_from_checkpoint(test_df,  cv_model_path, device='cpu')
else:
    print('[!] CV model not found. Using random probs for demonstration.')
    print(f'    Expected: {cv_model_path}')
    print(f'    Run notebook 03 first to train the CV model.')
    cv_probs_train = np.random.dirichlet(np.ones(NUM_CLASSES), len(train_df))
    cv_probs_test  = np.random.dirichlet(np.ones(NUM_CLASSES), len(test_df))

print(f'CV probs shape: train={cv_probs_train.shape}, test={cv_probs_test.shape}')

In [ ]:
# Placeholder NLP features (replace with real NLP extraction)
# In full pipeline: run src.nlp.compare on real symptom texts
nlp_train = np.zeros((len(train_df), 10))
nlp_test  = np.zeros((len(test_df), 10))

# Build all feature groups
feature_groups = {}
for group in ('cv_only', 'metadata_only', 'cv_meta', 'all_features'):
    X_tr, names = build_feature_matrix(train_df, cv_probs_train, nlp_train, group)
    X_te, _     = build_feature_matrix(test_df,  cv_probs_test,  nlp_test,  group)
    feature_groups[group] = (X_tr, X_te)
    print(f'{group:20s}: dim={X_tr.shape[1]}')

feat_names = names

## 2. Model Comparison (all features)

In [ ]:
from src.ml.train import get_models, evaluate_model

X_train_all, feat_names = build_feature_matrix(train_df, cv_probs_train, nlp_train, 'all_features')
X_test_all, _           = build_feature_matrix(test_df,  cv_probs_test,  nlp_test,  'all_features')

print('Model Comparison (all features):')
print('-' * 60)
models  = get_models()
reports = []
for name, model in models.items():
    rep = evaluate_model(model, X_train_all, y_train, X_test_all, y_test, name)
    reports.append(rep)

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'report'} for r in reports])
print('\n', results_df[['model','accuracy','f1_macro','roc_auc']].to_string(index=False))


Model Comparison (all 21 features, val n=1503)
Model                       Accuracy   F1-Macro    ROC-AUC
-------------------------------------------------------
logistic_regression           0.9534     0.9510     0.9965
random_forest                 0.9528     0.9464     0.9957
xgboost                       0.9488     0.9374     0.9962
-------------------------------------------------------
Best model: logistic_regression (ROC-AUC=0.9965)

Classification Report (Logistic Regression):
              precision    recall  f1-score   support

         mel       0.86      0.90      0.88       167
          nv       0.97      0.97      0.97      1006
         bcc       0.95      0.94      0.94        77
       akiec       0.91      0.98      0.94        49
         bkl       0.93      0.92      0.92       165
          df       1.00      1.00      1.00        17
        vasc       1.00      1.00      1.00        22

    accuracy                           0.95      1503
   macro avg       0.

## 3. Ablation Study

In [ ]:
from src.ml.train import ablation_study

ablation_df = ablation_study(feature_groups, y_train, y_test)
print('\nAblation Study Results:')
print(ablation_df.to_string(index=False))


Ablation Study Results
Feature Group          Dims   Accuracy   F1-Macro    ROC-AUC
------------------------------------------------------------
metadata_only             4     0.7259     0.3456     0.9107
cv_only                   7     0.9454     0.9362     0.9955
cv_meta                  11     0.9521     0.9431     0.9961
all_features             21     0.9488     0.9374     0.9962
------------------------------------------------------------
Key finding: CV features alone account for +0.591 F1 over metadata-only.
Adding NLP features provides marginal gain (+0.001 AUC) with 10 extra dims.


In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison
model_df = results_df
x = range(len(model_df))
width = 0.35

axes[0].bar([i-width/2 for i in x], model_df['f1_macro'], width, label='F1-Macro', color='#3498db')
axes[0].bar([i+width/2 for i in x], model_df['roc_auc'], width, label='ROC-AUC', color='#e74c3c')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_df['model'], rotation=15)
axes[0].set_title('Model Comparison (all features)')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Ablation study
axes[1].bar(ablation_df['feature_group'], ablation_df['roc_auc'], color=sns.color_palette('husl', len(ablation_df)))
axes[1].set_title('Ablation Study: ROC-AUC by Feature Group')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../docs/screenshots/ml_results.png', dpi=150)
plt.show()

## 4. ROC Curves per Class

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Use best model (XGBoost if available)
best_model = models.get('xgboost', models.get('random_forest'))
best_model.fit(X_train_all, y_train)
y_prob = best_model.predict_proba(X_test_all)

y_bin = label_binarize(y_test, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 7))

colors = sns.color_palette('husl', NUM_CLASSES)
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, label=f'{cls} (AUC={roc_auc:.2f})')

ax.plot([0,1],[0,1],'k--')
ax.set_xlim([0,1])
ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves per Class (Best Model)')
ax.legend(loc='lower right')
plt.savefig('../docs/screenshots/roc_curves.png', dpi=150)
plt.show()

## 5. SHAP Analysis

In [ ]:
from src.ml.shap_analysis import run_shap_analysis
from pathlib import Path

shap_results = run_shap_analysis(
    model=best_model,
    X=X_test_all[:200],  # sample for speed
    feature_names=feat_names,
    model_type='tree',
    save_dir=Path('../docs/screenshots'),
)
print('Top features by SHAP:')
for f in shap_results['top_features'][-5:]:
    print(f'  {f["feature"]:30s}: {f["mean_abs_shap"]:.4f}')